# Análisis de los experimentos de K-Means

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent

ruta_resultados = (
    PROJECT_ROOT
    / "resultados"
    / "registro_experimentos.parquet"
)

df_resultados = pd.read_parquet(ruta_resultados)

print(f"Filas registradas: {len(df_resultados):,}")
print(f"Columnas: {df_resultados.shape[1]}")

Filas registradas: 1,080
Columnas: 36


In [3]:
# revisar identificadores duplicados
columnas_metricas = [
    "mse_train",
    "rss_train",
    "mse_val",
    "rss_val",
    "purity_val",
    "davies_bouldin_val",
    "silhouette_val",
    "ami_val",
]

print(
    "Experiment ID duplicados:",
    df_resultados["experiment_id"].duplicated().sum(),
)

print("\nEstados:")
display(df_resultados["estado"].value_counts(dropna=False))

print("\nValores faltantes en métricas:")
display(
    df_resultados[columnas_metricas]
    .isna()
    .sum()
    .rename("n_faltantes")
    .to_frame()
)

Experiment ID duplicados: 0

Estados:


estado
OK    1080
Name: count, dtype: int64


Valores faltantes en métricas:


,n_faltantes
mse_train,0
rss_train,0
mse_val,0
rss_val,0
purity_val,0
davies_bouldin_val,0
silhouette_val,0
ami_val,0


In [4]:
columnas_configuracion = [
    "dataset",
    "normalizacion",
    "K",
    "seed_modelo",
    "inicializacion",
]

repeticiones = (
    df_resultados
    .groupby(columnas_configuracion)
    .size()
)

print(f"Combinaciones únicas: {len(repeticiones):,}")
print(f"Mínimo de registros por combinación: {repeticiones.min()}")
print(f"Máximo de registros por combinación: {repeticiones.max()}")

print("\nValores encontrados en cada variable:")

for columna in columnas_configuracion:
    valores = sorted(
        df_resultados[columna].unique().tolist()
    )
    print(f"{columna}: {valores}")

Combinaciones únicas: 1,080
Mínimo de registros por combinación: 1
Máximo de registros por combinación: 1

Valores encontrados en cada variable:
dataset: ['interpolacion_100', 'interpolacion_50', 'medianas_fase_100', 'medianas_fase_50']
normalizacion: ['centered', 'minmax', 'zscore']
K: [2, 3, 4, 5, 6, 7, 8, 9, 10]
seed_modelo: [0, 1, 2, 3, 4]
inicializacion: ['k-means++', 'random']


In [5]:
# Revisar números finitos y cantidad de iteraciones
metricas_numericas = df_resultados[
    columnas_metricas
].to_numpy(dtype=np.float64)

print(
    "Valores no finitos en métricas:",
    (~np.isfinite(metricas_numericas)).sum(),
)

print(
    "Experimentos que alcanzaron max_iter:",
    df_resultados["alcanzo_max_iter"].sum(),
)

print(
    "Tiempos de entrenamiento no positivos:",
    (
        df_resultados["tiempo_entrenamiento_segundos"] <= 0
    ).sum(),
)

print(
    "Tiempos totales no positivos:",
    (
        df_resultados["tiempo_total_segundos"] <= 0
    ).sum(),
)

Valores no finitos en métricas: 0
Experimentos que alcanzaron max_iter: 0
Tiempos de entrenamiento no positivos: 0
Tiempos totales no positivos: 0


In [6]:
import sys
from itertools import combinations

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from metricas import ari_metric

In [7]:
filtro = (
    (df_resultados["dataset"] == "interpolacion_50")
    & (df_resultados["normalizacion"] == "minmax")
    & (df_resultados["K"] == 3)
    & (df_resultados["inicializacion"] == "k-means++")
)

ejecuciones = (
    df_resultados.loc[filtro]
    .sort_values("seed_modelo")
)

print(f"Ejecuciones encontradas: {len(ejecuciones)}")
display(
    ejecuciones[
        ["experiment_id", "seed_modelo"]
    ]
)

Ejecuciones encontradas: 5


,experiment_id,seed_modelo
11,793d1ae66d7babfd,0
13,7ecdc1d0b9605bb3,1
15,349ed45927f58455,2
17,2c84253c121ddddd,3
19,60892420d2db8d90,4


In [8]:
comparaciones_ari = []

for (_, ejecucion_a), (_, ejecucion_b) in combinations(
    ejecuciones.iterrows(),
    2,
):
    carpeta_a = (
        PROJECT_ROOT
        / "resultados"
        / "experimentos"
        / ejecucion_a["experiment_id"]
    )

    carpeta_b = (
        PROJECT_ROOT
        / "resultados"
        / "experimentos"
        / ejecucion_b["experiment_id"]
    )

    ids_a = np.load(carpeta_a / "ids_val.npy")
    ids_b = np.load(carpeta_b / "ids_val.npy")

    if not np.array_equal(ids_a, ids_b):
        raise ValueError(
            "Las ejecuciones no contienen las mismas curvas "
            "en el mismo orden."
        )

    grupos_a = np.load(carpeta_a / "grupos_val.npy")
    grupos_b = np.load(carpeta_b / "grupos_val.npy")

    ari = ari_metric(grupos_a, grupos_b)

    comparaciones_ari.append({
        "seed_a": ejecucion_a["seed_modelo"],
        "seed_b": ejecucion_b["seed_modelo"],
        "ari": ari,
    })

df_ari_prueba = pd.DataFrame(comparaciones_ari)

display(df_ari_prueba)

,seed_a,seed_b,ari
0,0,1,0.999847
1,0,2,0.995694
2,0,3,0.999847
3,0,4,0.995694
4,1,2,0.995846
5,1,3,1.000000
6,1,4,0.995846
7,2,3,0.995846
8,2,4,1.000000
9,3,4,0.995846


In [9]:
resumen_ari_prueba = {
    "ari_mean": df_ari_prueba["ari"].mean(),
    "ari_std": df_ari_prueba["ari"].std(ddof=0),
    "ari_min": df_ari_prueba["ari"].min(),
}

print(resumen_ari_prueba)

{'ari_mean': np.float64(0.9974466198066041), 'ari_std': np.float64(0.002023685443864748), 'ari_min': np.float64(0.9956937419808849)}


In [10]:
columnas_base = [
    "modelo",
    "dataset",
    "normalizacion",
    "K",
    "inicializacion",
]

comparaciones_ari = []

for valores, ejecuciones_grupo in df_resultados.groupby(
    columnas_base
):
    ejecuciones_grupo = ejecuciones_grupo.sort_values(
        "seed_modelo"
    )

    asignaciones = []
    ids_referencia = None

    for _, ejecucion in ejecuciones_grupo.iterrows():
        carpeta = (
            PROJECT_ROOT
            / "resultados"
            / "experimentos"
            / ejecucion["experiment_id"]
        )

        ids = np.load(carpeta / "ids_val.npy")
        grupos = np.load(carpeta / "grupos_val.npy")

        if ids_referencia is None:
            ids_referencia = ids
        elif not np.array_equal(ids_referencia, ids):
            raise ValueError(
                "Las ejecuciones de una configuración no contienen "
                "las mismas curvas en el mismo orden."
            )

        asignaciones.append({
            "experiment_id": ejecucion["experiment_id"],
            "seed": ejecucion["seed_modelo"],
            "grupos": grupos,
        })

    for ejecucion_a, ejecucion_b in combinations(
        asignaciones,
        2,
    ):
        comparaciones_ari.append({
            **dict(zip(columnas_base, valores)),
            "experiment_id_a": ejecucion_a["experiment_id"],
            "experiment_id_b": ejecucion_b["experiment_id"],
            "seed_a": ejecucion_a["seed"],
            "seed_b": ejecucion_b["seed"],
            "ari": ari_metric(
                ejecucion_a["grupos"],
                ejecucion_b["grupos"],
            ),
        })

df_ari_detalle = pd.DataFrame(comparaciones_ari)

print(f"Comparaciones ARI calculadas: {len(df_ari_detalle):,}")
display(df_ari_detalle.head())

Comparaciones ARI calculadas: 2,160


,modelo,dataset,normalizacion,K,inicializacion,experiment_id_a,experiment_id_b,seed_a,seed_b,ari
0,kmeans,interpolacion_100,centered,2,k-means++,2447a71c84d09674,7c80f8d52230a617,0,1,0.999806
1,kmeans,interpolacion_100,centered,2,k-means++,2447a71c84d09674,95f74f3504ede5c3,0,2,0.999806
2,kmeans,interpolacion_100,centered,2,k-means++,2447a71c84d09674,191b7da34b9584a1,0,3,0.998251
3,kmeans,interpolacion_100,centered,2,k-means++,2447a71c84d09674,4251a3ec5a43903b,0,4,0.999806
4,kmeans,interpolacion_100,centered,2,k-means++,7c80f8d52230a617,95f74f3504ede5c3,1,2,1.000000


In [11]:
df_ari_resumen = (
    df_ari_detalle
    .groupby(columnas_base)
    .agg(
        ari_mean=(
            "ari",
            "mean",
        ),
        ari_std=(
            "ari",
            lambda valores: valores.std(ddof=0),
        ),
        ari_min=(
            "ari",
            "min",
        ),
    )
    .reset_index()
)

print(
    f"Configuraciones resumidas: "
    f"{len(df_ari_resumen):,}"
)

display(df_ari_resumen.head())

Configuraciones resumidas: 216


,modelo,dataset,normalizacion,K,inicializacion,ari_mean,ari_std,ari_min
0,kmeans,interpolacion_100,centered,2,k-means++,0.999300,0.000744,0.998251
1,kmeans,interpolacion_100,centered,2,random,0.999145,0.000646,0.998445
2,kmeans,interpolacion_100,centered,3,k-means++,0.996061,0.002816,0.992961
3,kmeans,interpolacion_100,centered,3,random,0.997551,0.002652,0.993947
4,kmeans,interpolacion_100,centered,4,k-means++,0.989573,0.007613,0.980752


In [12]:
ESTABILIDAD_DIR = (
    PROJECT_ROOT
    / "resultados"
    / "estabilidad"
)

ESTABILIDAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

df_ari_detalle.to_parquet(
    ESTABILIDAD_DIR / "ari_kmeans_detalle.parquet",
    index=False,
)

df_ari_resumen.to_csv(
    ESTABILIDAD_DIR / "ari_kmeans_resumen.csv",
    index=False,
)

df_ari_resumen.to_parquet(
    ESTABILIDAD_DIR / "ari_kmeans_resumen.parquet",
    index=False,
)

print(f"Resultados guardados en: {ESTABILIDAD_DIR}")

Resultados guardados en: C:\Users\adanm\Code\tesis\resultados\estabilidad
